# Relative Effectiveness
Generate habitat condition and loss metrics within each cell, and calculate relative effectiveness metrics by comparing scores for matched treatment and control cells.

In [ ]:
# Select site by WDPAID (one from the list of 30 in variables.py)
site_id = 1543

In [ ]:
from pathlib import Path
import os
import sys
import ee
import geemap
import geopandas as gpd

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    ANALYSIS_END_YR,
)

from absolute_effectiveness.site_selector import SiteSelector
from absolute_effectiveness.data_processor import DataProcessor
from relative_effectiveness.metrics_per_cell import (
    RelativeHabitatConditionAnalyzer,
    RelativeHabitatLossAnalyzer,
)

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()
processor = DataProcessor.from_gee_defaults()
condition_analyzer = RelativeHabitatConditionAnalyzer()
loss_analyzer = RelativeHabitatLossAnalyzer()

In [ ]:
# Load matched_grids and derive site-specific context

matched_grids_gdf = gpd.read_parquet(f"data/matched_grids_{site_id}.parquet").to_crs(epsg=4326)
matched_grids = geemap.geopandas_to_ee(matched_grids_gdf)

test_sites = site_selector.get_test_sites()
START_YR = site_selector.set_start_yr(test_sites, site_id)
site_geom = site_selector.get_site_geom(test_sites, site_id)

site_selector.check_start_yr(START_YR)

In [ ]:
# Process all input datasets, scoped to the bounds of the matched cells
# (covers PA + control buffer, since control cells live outside the PA)
GLC_processed = processor.process_glc(matched_grids, START_YR)
GPW_processed = processor.process_gpw(START_YR)
NFW_processed = processor.process_nfw(matched_grids)
HGFC_processed = processor.process_hgfc(START_YR)

In [ ]:
# Build habitat / intactness rasters and score Habitat Extent, Intactness, and Condition per cell
habitat_raster = condition_analyzer.get_habitat_raster(
    GLC_processed, HGFC_processed, GPW_processed, NFW_processed
)
exp_kernel = condition_analyzer.build_kernel()
intactness_raster = condition_analyzer.get_intactness_raster(
    habitat_raster, matched_grids.geometry(), exp_kernel
)

scored = condition_analyzer.calc_extent_score_per_cell(habitat_raster, matched_grids)
scored = condition_analyzer.calc_intactness_score_per_cell(intactness_raster, scored)
scored = condition_analyzer.calc_condition_score_per_cell(scored)

In [ ]:
# Build habitat-loss rasters and score Habitat Loss per cell
habitat_loss_raster, habitat_start_raster = loss_analyzer.get_habitat_loss_raster(
    GLC_processed, GPW_processed, HGFC_processed, START_YR
)
scored = loss_analyzer.calc_loss_score_per_cell(
    habitat_loss_raster, habitat_start_raster, scored
)

print(f"Site ID: {site_id}")
print(f"Analysis Period: {START_YR} - {ANALYSIS_END_YR}")
print(f"Scored cells: {scored.size().getInfo()}")
print("\nFirst feature properties:")
print(scored.first().getInfo()["properties"])

## Calculate relative effectiveness
Calculate differences in effectiveness metrics between matched treatment and control cells, and aggregate relative metrics for the PA.

In [ ]:
import pandas as pd

# Import match table from ps_model.ipynb
match_table = pd.read_parquet(f"data/match_table_{site_id}.parquet")

mt = match_table.copy()
mt["treat_cell_id"] = mt["treat_cell_id"].astype(int)
mt["control_cell_id"] = mt["control_cell_id"].astype(int)

# Convert cell scores to GeoDataFrame
score_gdf = geemap.ee_to_gdf(scored)
score_cols = ["extent_score", "intactness_score", "condition_score", "loss_score"]
scores = score_gdf[["cell_ID"] + score_cols].drop_duplicates("cell_ID").copy()
scores["cell_ID"] = scores["cell_ID"].astype(int)

# Join cell scores to match table by cell ID
treat_renamed = {"cell_ID": "treat_cell_id", **{c: f"treat_{c}" for c in score_cols}}
ctrl_renamed = {"cell_ID": "control_cell_id", **{c: f"control_{c}" for c in score_cols}}

match_df = mt.merge(
    scores.rename(columns=treat_renamed), on="treat_cell_id", how="left"
).merge(scores.rename(columns=ctrl_renamed), on="control_cell_id", how="left")

# Calculate pairwise differences between treatment and control scores for matched cells
for col in score_cols:
    match_df[f"{col}_diff"] = match_df[f"treat_{col}"] - match_df[f"control_{col}"]

match_df.head()

In [ ]:
# Calculate average differences in 2 steps to avoid weighting cells with more matches higher

# Average differences across control cells per treatment cell
treat_means = match_df.groupby("treat_cell_id")[["extent_score_diff", "intactness_score_diff", "condition_score_diff", "loss_score_diff"]].mean()

# Average differences across treatment cells for the whole PA
pa_relative_extent = treat_means["extent_score_diff"].mean()
pa_relative_intactness = treat_means["intactness_score_diff"].mean()
pa_relative_condition = treat_means["condition_score_diff"].mean()
pa_relative_loss = treat_means["loss_score_diff"].mean()

print("PA Relative Extent Score:", pa_relative_extent)
print("PA Relative Intactness Score:", pa_relative_intactness)
print("PA Relative Condition Score:", pa_relative_condition)
print("PA Relative Loss Score:", pa_relative_loss)

## Visualization

In [ ]:
from utils.variables import GLC_PALETTE

score_palette = ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"]
score_viz = {"min": 0, "max": 1, "palette": score_palette}

def score_image(fc, prop):
    """Rasterize a numeric per-feature score so it can be displayed with a palette."""
    return fc.reduceToImage(properties=[prop], reducer=ee.Reducer.first())

Map = geemap.Map()
Map.add_basemap("CartoDB.Positron")

Map.addLayer(habitat_raster, {"palette": GLC_PALETTE}, "Habitat Extent", 0)
Map.addLayer(intactness_raster, {"min": 0, "max": 1, "palette": ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"]}, "Habitat Intactness", 0)

Map.addLayer(site_geom, {"color": "yellow"}, "PA boundary", False)
Map.addLayer(matched_grids, {"color": "white"}, "Matched cells (outline)", False)

Map.addLayer(score_image(scored, "extent_score"), score_viz, "Extent Score")
Map.addLayer(score_image(scored, "intactness_score"), score_viz, "Intactness Score")
Map.addLayer(score_image(scored, "condition_score"), score_viz, "Condition Score")
Map.addLayer(score_image(scored, "loss_score"), score_viz, "Loss Score")

Map.centerObject(matched_grids)

Map